# FabricCare V3 · Fabric classification

Train MobileNetV2, GLCM + Decision Tree, and GLCM + SVM on the same six classes and the published V3 garment-grouped splits. The notebook selects settings on validation and reserves test images for the final comparison.

1. Open [FabriCare_Dataset_V3](https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8) and add a shortcut with that name directly under **My Drive**.
2. Open this notebook in Colab and select a GPU runtime. Run cells in order and authorize the Drive mount.
3. New exports appear under **My Drive → Fabricare_training_v3 → timestamp**. The original Drive images and published V3 models remain available as references.

The notebook checks the pinned V3 split revision, labels, duplicate hashes and group isolation before training. The image preparation matches the scanner's 224px gray padding and EXIF orientation. The output is evaluated separately before deployment.


In [ ]:
%pip -q install scikit-learn==1.6.1 scikit-image==0.26.0 joblib==1.5.3


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv
import hashlib
import io
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps

V3_URL = "https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8"
V3_ROOT = Path("/content/drive/MyDrive/FabriCare_Dataset_V3")
MANIFEST_SHA256 = "f164042dcb4a0b4876fc5474532df1bb7b25403703e4a70852f19ada64c287ea"
SPLITS = ("train", "valid", "test")
CLASS_NAMES = ["cotton", "denim", "linen", "polyester", "silk", "wool"]
SEED = 42

def read_v3_manifest(root):
    manifest = root / "reports" / "split_manifest_70_20_10.csv"
    if not manifest.is_file():
        raise FileNotFoundError(f"Open {V3_URL} and add its shortcut to My Drive as {root.name}.")
    data = manifest.read_bytes()
    if hashlib.sha256(data).hexdigest() != MANIFEST_SHA256:
        raise ValueError("The split manifest differs from the audited V3 revision. Verify the folder and split file.")
    rows = list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"), newline="")))
    counts, groups, digests = Counter(), defaultdict(set), set()
    for row in rows:
        split, label, filename = row["split"], row["fabric"], row["filename"]
        if split not in SPLITS or label not in CLASS_NAMES or not filename or Path(filename).name != filename:
            raise ValueError(f"Invalid V3 manifest entry: {row}")
        digest = row["sha256"]
        if digest in digests:
            raise ValueError(f"Duplicate content SHA-256 in the split manifest: {filename}")
        if not row["group_id"]:
            raise ValueError(f"Missing garment group: {filename}")
        digests.add(digest)
        groups[row["group_id"]].add(split)
        counts[split, label] += 1
    if any(len(value) > 1 for value in groups.values()):
        raise ValueError("A garment group appears in multiple splits.")
    for split in SPLITS:
        if any(counts[split, label] == 0 for label in CLASS_NAMES):
            raise ValueError(f"Missing fabric class in {split}.")
        print(split, {label: counts[split, label] for label in CLASS_NAMES})
    return rows

def stage_v3(root, records, target, *, padded):
    staged = []
    for record in records:
        label, split = record["fabric"], record["split"]
        source = root / "raw" / label / record["filename"]
        content = source.read_bytes()
        if hashlib.sha256(content).hexdigest() != record["sha256"]:
            raise ValueError(f"Image bytes differ from the V3 manifest: {source}")
        parent = target / split / label
        parent.mkdir(parents=True, exist_ok=True)
        if padded:
            destination = parent / (record["sha256"] + ".png")
            with Image.open(io.BytesIO(content)) as raw:
                # Normalize archival originals before submitting them to the classifier.
                rgb = ImageOps.exif_transpose(raw).convert("RGB")
                ImageOps.pad(rgb, (224, 224), method=Image.Resampling.BILINEAR,
                             color=(127, 127, 127)).save(destination)
        else:
            destination = parent / record["filename"]
            destination.write_bytes(content)
        staged.append((str(destination), CLASS_NAMES.index(label)))
    return staged


In [ ]:
from datetime import datetime, timezone
import importlib.metadata
import json
import shutil
import tempfile

import joblib
import numpy as np
import sklearn
import tensorflow as tf
from skimage.feature import graycomatrix, graycoprops
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

if not tf.config.list_physical_devices("GPU"):
    print("No GPU detected. Training can run on CPU and will take longer.")
if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Restart the runtime after installing the scanner-compatible dependencies.")
tf.keras.utils.set_random_seed(SEED)
rows = read_v3_manifest(V3_ROOT)
run = Path("/content/drive/MyDrive/Fabricare_training_v3") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run.mkdir(parents=True, exist_ok=False)
stage = Path(tempfile.mkdtemp(prefix="fabricare-v3-", dir="/content"))
samples = {
    split: stage_v3(V3_ROOT, [r for r in rows if r["split"] == split], stage, padded=True)
    for split in ("train", "valid")
}
print("New run:", run)


## Train and select MobileNetV2 on validation data


In [ ]:
def dataset(records, shuffle=False):
    paths, labels = zip(*records)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), np.asarray(labels, dtype=np.int32)))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    def decode(path, label):
        image = tf.io.decode_png(tf.io.read_file(path), channels=3)
        return tf.cast(tf.ensure_shape(image, (224, 224, 3)), tf.float32), label
    return ds.map(decode, num_parallel_calls=4).batch(32).prefetch(2)

backbone = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
backbone.trainable = False
inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.layers.RandomFlip("horizontal", seed=SEED)(inputs)
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(x)
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
network = tf.keras.Model(inputs, tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x))
train_data = dataset(samples["train"], shuffle=True)
valid_data = dataset(samples["valid"])
valid_labels = [label for _, label in samples["valid"]]
train_counts = Counter(label for _, label in samples["train"])
weights = {i: len(samples["train"]) / (len(CLASS_NAMES) * n) for i, n in train_counts.items()}

def train_stage(name, rate, epochs):
    local = stage / (name + ".keras")
    network.compile(optimizer=tf.keras.optimizers.Adam(rate),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = network.fit(train_data, validation_data=valid_data, class_weight=weights, epochs=epochs,
                         callbacks=[tf.keras.callbacks.ModelCheckpoint(str(local), monitor="val_loss", save_best_only=True),
                                    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                                     restore_best_weights=True)])
    shutil.copy2(local, run / local.name)
    (run / (name + "_history.json")).write_text(json.dumps(history.history, indent=2))
    predicted = network.predict(valid_data, verbose=0).argmax(axis=1)
    return f1_score(valid_labels, predicted, average="macro", zero_division=0)

validation = {"stage1": train_stage("mobilenetv2_stage1", 1e-3, 10)}
backbone.trainable = True
for i, layer in enumerate(backbone.layers):
    layer.trainable = i >= len(backbone.layers) - 30 and not isinstance(layer, tf.keras.layers.BatchNormalization)
validation["stage2"] = train_stage("mobilenetv2_stage2", 1e-5, 12)
selected_stage = max(validation, key=validation.get)
cnn_path = run / "mobilenetv2_stage2_best.keras"  # Stable filename expected by the API.
shutil.copy2(run / ("mobilenetv2_" + selected_stage + ".keras"), cnn_path)
network = tf.keras.models.load_model(cnn_path, compile=False)
(run / "class_names.json").write_text(json.dumps(CLASS_NAMES, indent=2))
print("Validation macro F1 by CNN stage:", validation, "Selected:", selected_stage)


## Train GLCM + Decision Tree and GLCM + SVM


In [ ]:
GLCM_PROPERTIES = ("contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM")

def glcm72(records):
    vectors = []
    for path, _ in records:
        with Image.open(path) as image:
            gray = ImageOps.grayscale(image).resize((128, 128), Image.Resampling.BICUBIC)
            matrix = graycomatrix(np.asarray(gray, dtype=np.uint8) // 8, [1, 2, 4],
                                   [0, np.pi/4, np.pi/2, 3*np.pi/4],
                                   levels=32, symmetric=True, normed=True)
            vectors.append(np.nan_to_num(np.concatenate([graycoprops(matrix, prop).ravel() for prop in GLCM_PROPERTIES]),
                                         nan=0, posinf=0, neginf=0))
    return np.stack(vectors)

train_x = glcm72(samples["train"])
train_y = [label for _, label in samples["train"]]
valid_x = glcm72(samples["valid"])
candidates = {
    "glcm_decision_tree": [DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=SEED)
                           for depth in (8, 16, None)],
    "glcm_svm": [make_pipeline(StandardScaler(), SVC(C=c, class_weight="balanced")) for c in (1, 5, 10)],
}
selected = {}
for name, models in candidates.items():
    trials = []
    for candidate in models:
        candidate.fit(train_x, train_y)
        score = f1_score(valid_labels, candidate.predict(valid_x), average="macro", zero_division=0)
        trials.append((score, candidate))
    score, model = max(trials, key=lambda row: row[0])
    validation[name] = float(score)
    selected[name] = model
    joblib.dump(model, run / (name + ".joblib"))
    print(name, "validation macro F1:", score)


## Final comparison on the V3 test split


In [ ]:
test = stage_v3(V3_ROOT, [r for r in rows if r["split"] == "test"], stage, padded=True)
test_y = [label for _, label in test]
test_x = glcm72(test)
predictions = {"mobilenetv2": network.predict(dataset(test), verbose=0).argmax(axis=1)}
predictions.update({name: model.predict(test_x) for name, model in selected.items()})
label_ids = list(range(len(CLASS_NAMES)))
metrics = {
    name: {
        "accuracy": accuracy_score(test_y, guess),
        "macro_f1": f1_score(test_y, guess, labels=label_ids, average="macro", zero_division=0),
        "per_class": classification_report(test_y, guess, labels=label_ids, target_names=CLASS_NAMES,
                                          output_dict=True, zero_division=0),
        "confusion_matrix_actual_by_predicted": confusion_matrix(test_y, guess, labels=label_ids).tolist(),
    }
    for name, guess in predictions.items()
}
artifacts = ("mobilenetv2_stage2_best.keras", "glcm_decision_tree.joblib", "glcm_svm.joblib", "class_names.json")
report = {
    "dataset_url": V3_URL, "split_manifest_sha256": MANIFEST_SHA256, "seed": SEED,
    "class_order": CLASS_NAMES, "test_images": len(test), "validation_macro_f1": validation,
    "selected_cnn_stage": selected_stage, "test": metrics,
    "versions": {name: importlib.metadata.version(name) for name in ("tensorflow", "scikit-learn", "scikit-image", "pillow", "joblib")},
    "artifact_sha256": {name: hashlib.sha256((run / name).read_bytes()).hexdigest() for name in artifacts},
}
(run / "training_report.json").write_text(json.dumps(report, indent=2))
shutil.copy2(V3_ROOT / "reports" / "split_manifest_70_20_10.csv", run)
for name, result in metrics.items():
    print(name, "accuracy:", round(result["accuracy"], 4), "macro F1:", round(result["macro_f1"], 4))
    print("Recall by fabric:", {label: round(result["per_class"][label]["recall"], 4) for label in CLASS_NAMES})
print("Exports:", run)
